# Berka Czech Bank Dataset - End-to-End Exploration & Cleaning

In [9]:
import os
import pandas as pd
import numpy as np

RAW_DIR = 'raw-data'
CLEAN_DIR = 'cleaned-data'

os.makedirs(CLEAN_DIR, exist_ok=True)
pd.set_option('display.max_columns', None)

## Step 1: Import Raw Files
First check how the raw files are delimited before loading all 8 tables.

In [23]:
# peek at the first two lines of account.csv to verify separator
with open(f'{RAW_DIR}/account.csv', 'r') as f:
    for _ in range(2):
        print(f.readline().strip())

"account_id";"district_id";"frequency";"date"
576;55;"POPLATEK MESICNE";930101


In [24]:
# files are semicolon-delimited (';')
account = pd.read_csv(f'{RAW_DIR}/account.csv', sep=';')
card = pd.read_csv(f'{RAW_DIR}/card.csv', sep=';')
client = pd.read_csv(f'{RAW_DIR}/client.csv', sep=';')
disp = pd.read_csv(f'{RAW_DIR}/disp.csv', sep=';')
district = pd.read_csv(f'{RAW_DIR}/district.csv', sep=';')
loan = pd.read_csv(f'{RAW_DIR}/loan.csv', sep=';')
order = pd.read_csv(f'{RAW_DIR}/order.csv', sep=';')
trans = pd.read_csv(f'{RAW_DIR}/trans.csv', sep=';', low_memory=False)

tables = {
    'account': account, 'card': card, 'client': client, 'disp': disp,
    'district': district, 'loan': loan, 'order': order, 'trans': trans
}

for name, df in tables.items():
    print(f"{name:<10} -> {df.shape[0]:>8,} rows | {df.shape[1]:>2} cols")

account    ->    4,500 rows |  4 cols
card       ->      892 rows |  4 cols
client     ->    5,369 rows |  3 cols
disp       ->    5,369 rows |  4 cols
district   ->       77 rows | 16 cols
loan       ->      682 rows |  7 cols
order      ->    6,471 rows |  6 cols
trans      -> 1,056,320 rows | 10 cols


## Step 2: Explore Data Types, Missing Values & Errors
Check column types, standard `NaN` nulls, hidden whitespace strings, and unexpected values across all tables.

In [25]:
# check data types and explicit NaN counts table by table
for name, df in tables.items():
    print(f"=== {name.upper()} ===")
    summary = pd.DataFrame({
        'dtype': df.dtypes,
        'null_count': df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
        'nunique': df.nunique()
    })
    print(summary, "\n")

=== ACCOUNT ===
              dtype  null_count  null_pct  nunique
account_id    int64           0       0.0     4500
district_id   int64           0       0.0       77
frequency    object           0       0.0        3
date          int64           0       0.0     1535 

=== CARD ===
          dtype  null_count  null_pct  nunique
card_id   int64           0       0.0      892
disp_id   int64           0       0.0      892
type     object           0       0.0        3
issued   object           0       0.0      607 

=== CLIENT ===
              dtype  null_count  null_pct  nunique
client_id     int64           0       0.0     5369
birth_number  int64           0       0.0     5019
district_id   int64           0       0.0       77 

=== DISP ===
             dtype  null_count  null_pct  nunique
disp_id      int64           0       0.0     5369
client_id    int64           0       0.0     5369
account_id   int64           0       0.0     4500
type        object           0       0.0   

In [12]:
# notice A12 and A15 in district loaded as 'object' instead of float/int
# filter rows where A12 or A15 cannot be parsed as a number
district[
    pd.to_numeric(district['A12'], errors='coerce').isna() |
    pd.to_numeric(district['A15'], errors='coerce').isna()
][['A1', 'A2', 'A3', 'A11', 'A12', 'A13', 'A15', 'A16']]

,A1,A2,A3,A11,A12,A13,A15,A16
68,69,Jesenik,north Moravia,8173,?,7.01,?,1358


In [13]:
# check string columns for hidden whitespace (' ') or Czech codes
print("account.frequency:", account['frequency'].value_counts(dropna=False).to_dict())
print("loan.status:", loan['status'].value_counts(dropna=False).to_dict())
print("order.k_symbol:", order['k_symbol'].value_counts(dropna=False).to_dict())
print("trans.type:", trans['type'].value_counts(dropna=False).to_dict())
print("trans.operation:", trans['operation'].value_counts(dropna=False).to_dict())
print("trans.k_symbol:", trans['k_symbol'].value_counts(dropna=False).to_dict())

account.frequency: {'POPLATEK MESICNE': 4167, 'POPLATEK TYDNE': 240, 'POPLATEK PO OBRATU': 93}
loan.status: {'C': 403, 'A': 203, 'D': 45, 'B': 31}
order.k_symbol: {'SIPO': 3502, ' ': 1379, 'UVER': 717, 'POJISTNE': 532, 'LEASING': 341}
trans.type: {'VYDAJ': 634571, 'PRIJEM': 405083, 'VYBER': 16666}
trans.operation: {'VYBER': 434918, 'PREVOD NA UCET': 208283, nan: 183114, 'VKLAD': 156743, 'PREVOD Z UCTU': 65226, 'VYBER KARTOU': 8036}
trans.k_symbol: {nan: 481881, 'UROK': 183114, 'SLUZBY': 155832, 'SIPO': 118065, ' ': 53433, 'DUCHOD': 30338, 'POJISTNE': 18500, 'UVER': 13580, 'SANKC. UROK': 1577}


In [14]:
# check client.birth_number structure (expected YYMMDD)
bn_months = client['birth_number'].astype(str).str.zfill(6).str.slice(2, 4).astype(int)
print("Month min:", bn_months.min(), "| Month max:", bn_months.max())
print("Rows with month > 12 (female +50 encoding):", (bn_months > 12).sum())
client.head(3)

Month min: 1 | Month max: 62
Rows with month > 12 (female +50 encoding): 2645


,client_id,birth_number,district_id
0,1,706213,18
1,2,450204,1
2,3,406009,1


In [15]:
# check duplicate primary keys across all tables
pk_cols = {
    'account': 'account_id', 'card': 'card_id', 'client': 'client_id',
    'disp': 'disp_id', 'district': 'A1', 'loan': 'loan_id',
    'order': 'order_id', 'trans': 'trans_id'
}

for name, pk in pk_cols.items():
    dups = tables[name][pk].duplicated().sum()
    print(f"{name:<10} ({pk}): {dups} duplicates")

account    (account_id): 0 duplicates
card       (card_id): 0 duplicates
client     (client_id): 0 duplicates
disp       (disp_id): 0 duplicates
district   (A1): 0 duplicates
loan       (loan_id): 0 duplicates
order      (order_id): 0 duplicates
trans      (trans_id): 0 duplicates


### Summary of Exploration Findings & Decisions
1. **Missing / Corrupt Values (`district`):** `Jesenik` (`A1 = 69`) has `'?'` in `A12` (1995 unemployment) and `A15` (1995 crimes). **Decision:** Do not drop the district (accounts and clients reference `district_id = 69`); coerce `'?'` to `NaN` and backfill with 1996 values (`A13`, `A16`).
2. **Missing Values (`trans` & `order`):**
   - `trans.operation` has `183,114` nulls — these correspond directly to `k_symbol == 'UROK'` (interest credited automatically by the bank, not a customer operation). **Decision:** Fill with `'Bank Interest / Auto'`.
   - `trans.k_symbol` (`481,881` nulls + `53,433` blank spaces `' '`) and `order.k_symbol` (`1,379` blank spaces `' '`) represent uncategorized transactions/orders. **Decision:** Strip whitespace and fill with `'Uncategorized'`.
   - `trans.bank` (`782,812` nulls) and `trans.account` (`760,931` nulls) are null whenever a transaction is internal (e.g., cash deposit/withdrawal) rather than an inter-bank transfer. **Decision:** Fill `bank` with `'Internal'` and keep `account` as nullable integer (`Int64`).
3. **Encoding & Language:** Decode `birth_number` (`+50` month = Female), translate Czech labels to English, and convert `YYMMDD` dates to proper `YYYY-MM-DD` dates.

---

## Step 3: Act on Findings — Clean & Handle Missing Data

### 3.1 District Table

In [16]:
district.columns = [
    'district_id', 'district_name', 'region', 'inhabitants',
    'mun_lt_499', 'mun_500_1999', 'mun_2000_9999', 'mun_gt_10000',
    'cities', 'urban_ratio', 'avg_salary', 'unemployment_95',
    'unemployment_96', 'entrepreneurs_per_1000', 'crimes_95', 'crimes_96'
]

# convert '?' in Jesenik (id 69) to NaN, then fill using 1996 figures
district['unemployment_95'] = pd.to_numeric(district['unemployment_95'], errors='coerce')
district['crimes_95'] = pd.to_numeric(district['crimes_95'], errors='coerce')

district['unemployment_95'] = district['unemployment_95'].fillna(district['unemployment_96'])
district['crimes_95'] = district['crimes_95'].fillna(district['crimes_96']).astype('Int64')

print("Remaining nulls in district:", district.isna().sum().sum())
district.loc[district['district_id'] == 69]

Remaining nulls in district: 0


,district_id,district_name,region,inhabitants,mun_lt_499,mun_500_1999,mun_2000_9999,mun_gt_10000,cities,urban_ratio,avg_salary,unemployment_95,unemployment_96,entrepreneurs_per_1000,crimes_95,crimes_96
68,69,Jesenik,north Moravia,42821,4,13,5,1,3,48.4,8173,7.01,7.01,124,1358,1358


### 3.2 Client Table

In [17]:
def decode_birth_number(bn):
    s = str(bn).zfill(6)
    year = 1900 + int(s[:2])
    month = int(s[2:4])
    day = int(s[4:6])
    
    if month > 50:
        gender = 'Female'
        month -= 50
    else:
        gender = 'Male'
        
    return pd.Series([f"{year}-{month:02d}-{day:02d}", gender])

client[['birth_date', 'gender']] = client['birth_number'].apply(decode_birth_number)
client['birth_date'] = pd.to_datetime(client['birth_date'])
client['age_1998'] = 1998 - client['birth_date'].dt.year

client = client[['client_id', 'district_id', 'gender', 'birth_date', 'age_1998']]
client.head()

,client_id,district_id,gender,birth_date,age_1998
0,1,18,Female,1970-12-13,28
1,2,1,Male,1945-02-04,53
2,3,1,Female,1940-10-09,58
3,4,5,Male,1956-12-01,42
4,5,5,Female,1960-07-03,38


### 3.3 Account, Disp & Card Tables

In [18]:
freq_map = {
    'POPLATEK MESICNE': 'Monthly',
    'POPLATEK TYDNE': 'Weekly',
    'POPLATEK PO OBRATU': 'After Transaction'
}

account['frequency'] = account['frequency'].map(freq_map)
account['created_date'] = pd.to_datetime('19' + account['date'].astype(str), format='%Y%m%d')
account = account[['account_id', 'district_id', 'frequency', 'created_date']]

card['issued_date'] = pd.to_datetime('19' + card['issued'].str.slice(0, 6), format='%Y%m%d')
card['type'] = card['type'].str.title()
card = card[['card_id', 'disp_id', 'type', 'issued_date']]

disp['type'] = disp['type'].str.title()

print("Nulls -> account:", account.isna().sum().sum(), "| card:", card.isna().sum().sum(), "| disp:", disp.isna().sum().sum())

Nulls -> account: 0 | card: 0 | disp: 0


### 3.4 Loan & Standing Order Tables

In [19]:
loan_status_map = {
    'A': 'Paid - No Issues',
    'B': 'Expired - Unpaid (Default)',
    'C': 'Running - Good Standing',
    'D': 'Running - In Debt (Default)'
}

loan['loan_date'] = pd.to_datetime('19' + loan['date'].astype(str), format='%Y%m%d')
loan['status_desc'] = loan['status'].map(loan_status_map)
loan['is_default'] = loan['status'].isin(['B', 'D']).astype(int)
loan = loan[['loan_id', 'account_id', 'loan_date', 'amount', 'duration', 'payments', 'status', 'status_desc', 'is_default']]

ksymbol_map = {
    'POJISTNE': 'Insurance',
    'SLUZBY': 'Statement Payment',
    'UROK': 'Interest Credited',
    'SANKC. UROK': 'Sanction Interest',
    'SIPO': 'Household',
    'DUCHOD': 'Old-Age Pension',
    'UVER': 'Loan Payment',
    'LEASING': 'Leasing'
}

# replace whitespace-only strings (' ') with NaN, map Czech codes, then fill remaining with 'Uncategorized'
order['k_symbol'] = (
    order['k_symbol']
    .str.strip()
    .replace('', np.nan)
    .map(ksymbol_map)
    .fillna('Uncategorized')
)
order['account_to'] = order['account_to'].astype(str).str.strip()

print("order.k_symbol counts:")
print(order['k_symbol'].value_counts())

order.k_symbol counts:
k_symbol
Household        3502
Uncategorized    1379
Loan Payment      717
Insurance         532
Leasing           341
Name: count, dtype: int64


### 3.5 Transactions Table (~1.05M rows)

In [20]:
type_map = {
    'PRIJEM': 'Credit',
    'VYDAJ': 'Withdrawal',
    'VYBER': 'Cash Withdrawal'
}

op_map = {
    'VYBER KARTOU': 'Credit Card Withdrawal',
    'VKLAD': 'Cash Deposit',
    'PREVOD Z UCTU': 'Collection from Bank',
    'VYBER': 'Cash Withdrawal',
    'PREVOD NA UCET': 'Remittance to Bank'
}

trans['trans_date'] = pd.to_datetime('19' + trans['date'].astype(str), format='%Y%m%d')
trans['type'] = trans['type'].map(type_map)

# nulls in operation occur when k_symbol == 'UROK' (automatic interest credit)
trans['operation'] = trans['operation'].map(op_map).fillna('Bank Interest / Auto')

# clean whitespace and nulls in k_symbol
trans['k_symbol'] = (
    trans['k_symbol']
    .astype(str)
    .str.strip()
    .replace({'': np.nan, 'nan': np.nan})
    .map(ksymbol_map)
    .fillna('Uncategorized')
)

# nulls in bank/account mean the transaction was internal (not an external bank transfer)
trans['bank'] = trans['bank'].fillna('Internal')
trans['account'] = trans['account'].astype('Int64')

trans = trans[['trans_id', 'account_id', 'trans_date', 'type', 'operation', 'amount', 'balance', 'k_symbol', 'bank', 'account']]
print("Remaining nulls in trans:")
print(trans.isna().sum())

Remaining nulls in trans:
trans_id           0
account_id         0
trans_date         0
type               0
operation          0
amount             0
balance            0
k_symbol           0
bank               0
account       760931
dtype: int64


## Step 4: Final Validation & Save Clean Data
Confirm foreign keys match across tables and save all 8 cleaned CSVs into `cleaned_data/`.

In [22]:
# verify referential integrity before saving
assert set(account['district_id']).issubset(set(district['district_id']))
assert set(client['district_id']).issubset(set(district['district_id']))
assert set(disp['client_id']).issubset(set(client['client_id']))
assert set(disp['account_id']).issubset(set(account['account_id']))
assert set(card['disp_id']).issubset(set(disp['disp_id']))
assert set(loan['account_id']).issubset(set(account['account_id']))
assert set(order['account_id']).issubset(set(account['account_id']))
assert set(trans['account_id']).issubset(set(account['account_id']))

clean_tables = {
    'district': district,
    'client': client,
    'account': account,
    'disp': disp,
    'card': card,
    'loan': loan,
    'order': order,
    'trans': trans
}

for name, df in clean_tables.items():
    path = f"{CLEAN_DIR}/{name}_clean.csv"
    df.to_csv(path, index=False)
    print(f"Saved {path:<30} -> {len(df):>8,} rows")

Saved cleaned-data/district_clean.csv ->       77 rows
Saved cleaned-data/client_clean.csv  ->    5,369 rows
Saved cleaned-data/account_clean.csv ->    4,500 rows
Saved cleaned-data/disp_clean.csv    ->    5,369 rows
Saved cleaned-data/card_clean.csv    ->      892 rows
Saved cleaned-data/loan_clean.csv    ->      682 rows
Saved cleaned-data/order_clean.csv   ->    6,471 rows
Saved cleaned-data/trans_clean.csv   -> 1,056,320 rows
